# Lab 4: Representation Learning

A representation is only as good as what a later task can read out of it. You
will write the three tools that test exactly that: the InfoNCE contrastive loss,
a k-nearest-neighbor probe, and reconstruction error. Then you compare a
contrastive encoder against an autoencoder, design an encoder to a budget, and
choose one augmentation to see which invariance it teaches.

**What you will do**

1. Compute the InfoNCE loss and confirm it on a fixed probe.
2. Read frozen features with a kNN probe.
3. Score an autoencoder with reconstruction error.
4. Design an encoder to a parameter budget, then change one augmentation.

The red **Your Task** banners mark the parts you complete. Everything else is
provided and ready to run.


## At a glance

| | |
|---|---|
| Stack | PyTorch and torchvision; a GPU runtime is recommended |
| You write | `info_nce_loss`, `knn_predict`, `reconstruction_mse` |
| You submit | Five values, R1 through R5, from the final cell |
| GPU runtime | A few minutes of training in quick mode; longer on the full split |


<div style="border-left:6px solid #A31F34;background:#fff5f6;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#A31F34;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Big picture</div>
<p style="margin:6px 0 0;">Two encoders can look equally good until you ask a downstream task to read their features. The kNN probe is that reader; use it to compare what the contrastive and reconstruction objectives actually organized.</p>
</div>


## What is graded

R1 through R3 check your three functions on fixed inputs. R4 accepts your encoder
design as long as its parameter count lands in the stated range. R5 is a
pass-or-fail flag on your experimental setup. A measured probe accuracy on real
images is never submitted; you read it to interpret your runs, and the graded
numbers are the same in quick and full mode.


<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">Lower reconstruction error does not mean better features. An autoencoder can preserve pixels a classifier ignores, so judge a representation by the probe you care about, not by how sharply it reconstructs.</p>
</div>


In [ ]:
from __future__ import annotations

import math
import random
from dataclasses import asdict, dataclass, replace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 7960
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def reset_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "axes.prop_cycle": plt.cycler(
        color=["#A31F34", "#1D4ED8", "#059669", "#B45309", "#7C3AED"]
    ),
})

reset_seed()
print(f"PyTorch {torch.__version__}; device = {DEVICE}")


In [ ]:
QUICK_MODE = True
DATA_ROOT = "./data"

TRAIN_PER_CLASS = 300
VALIDATION_PER_CLASS = 100 if QUICK_MODE else 300
DEFAULT_BATCH_SIZE = 256
EPOCHS = 3 if QUICK_MODE else 8
NUM_WORKERS = 0 if QUICK_MODE else 2

print(f"quick_mode={QUICK_MODE}, epochs={EPOCHS}")


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)
base_transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)]
)

split_source = datasets.CIFAR10(DATA_ROOT, train=True, download=True)


def stratified_split(targets, train_per_class, validation_per_class, seed):
    target_tensor = torch.as_tensor(targets)
    generator = torch.Generator().manual_seed(seed)
    train_indices, validation_indices = [], []
    for class_id in range(10):
        class_indices = torch.where(target_tensor == class_id)[0]
        order = torch.randperm(len(class_indices), generator=generator)
        class_indices = class_indices[order]
        needed = train_per_class + validation_per_class
        train_indices.extend(class_indices[:train_per_class].tolist())
        validation_indices.extend(
            class_indices[train_per_class:needed].tolist()
        )
    return sorted(train_indices), sorted(validation_indices)


train_indices, validation_indices = stratified_split(
    split_source.targets, TRAIN_PER_CLASS, VALIDATION_PER_CLASS, SEED
)
assert set(train_indices).isdisjoint(validation_indices)

image_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=base_transform, download=False
)
train_set = Subset(image_source, train_indices)
validation_set = Subset(image_source, validation_indices)


def seeded_loader(dataset, *, batch_size, shuffle, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
        generator=generator,
    )


def trainable_parameter_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("training examples:", len(train_indices),
      "validation examples:", len(validation_indices))


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 1</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>info_nce_loss</code></div>
</div>


Complete `info_nce_loss`. L2-normalize the anchor and positive features, form the
scaled similarity matrix `logits = anchor_n @ positive_n^T / temperature`, and
return the cross-entropy that treats each anchor's own positive (the diagonal) as
the correct class. In a batch of `N`, each anchor has one positive and `N - 1`
in-batch negatives.


<details style="border:1px solid #e5e7eb;border-radius:8px;padding:10px 14px;background:#f9fafb;color:#111827;margin:14px 0;">
<summary style="cursor:pointer;font-weight:600;">How do the labels encode the positives?</summary>
<div style="margin-top:8px;">After building the similarity matrix, the correct match for row <code>i</code> sits on the diagonal, at column <code>i</code>. So the labels are just <code>0, 1, ..., N-1</code>, and cross-entropy does the rest.</div>
</details>


In [ ]:
def info_nce_loss(anchor_features, positive_features, temperature):
    """Return the InfoNCE loss for matched anchor/positive rows."""
    # STUDENT TASK 1:
    #   normalize both feature sets along dim=1
    #   logits = anchor_n @ positive_n.t() / temperature
    #   labels = arange(N); return cross_entropy(logits, labels)
    raise NotImplementedError("Return the scalar InfoNCE loss.")


In [ ]:
probe_features = torch.eye(4)
infonce_probe_value = round(
    info_nce_loss(probe_features, probe_features, temperature=0.5).item(), 6
)
negatives_per_anchor = 256 - 1
infonce_probe_contract = (
    abs(infonce_probe_value - 0.340753) < 1e-4
    and negatives_per_anchor == 255
)
assert infonce_probe_contract, "Check normalization, temperature, and the diagonal labels."
print("Task 1 checks passed. InfoNCE probe loss:", infonce_probe_value,
      "| negatives per anchor in a 256-batch:", negatives_per_anchor)


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 2</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>knn_predict</code></div>
</div>


Complete `knn_predict`. Given frozen `train_features`/`train_labels` and
`query_features`, return each query's predicted label by majority vote over its
`k` nearest training neighbors (Euclidean distance). Use `torch.cdist` and
`torch.mode`.


In [ ]:
def knn_predict(train_features, train_labels, query_features, k=1):
    """Return predicted labels for each query by majority vote over k neighbors."""
    # STUDENT TASK 2:
    #   distances = cdist(query, train); take the k smallest per query
    #   gather their labels; return the per-query mode
    raise NotImplementedError("Return the predicted label for each query.")


In [ ]:
knn_train_features = torch.tensor([[0.0, 0.0], [10.0, 10.0]])
knn_train_labels = torch.tensor([0, 1])
knn_query_features = torch.tensor(
    [[0.0, 1.0], [9.0, 9.0], [1.0, 0.0], [5.0, 6.0]]
)
knn_query_labels = torch.tensor([0, 1, 0, 0])
knn_predictions = knn_predict(
    knn_train_features, knn_train_labels, knn_query_features, k=1
)
knn_probe_correct = int((knn_predictions == knn_query_labels).sum().item())
knn_probe_contract = (knn_probe_correct == 3)
assert knn_probe_contract, "Three of four queries fall nearest to their own class."
print("Task 2 checks passed. Correct kNN predictions:", knn_probe_correct)


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 3</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>reconstruction_mse</code></div>
</div>


Complete `reconstruction_mse`, the mean squared error between an autoencoder's
reconstruction and its input. The autoencoder objective below uses it directly.


In [ ]:
def reconstruction_mse(inputs, reconstructions):
    """Return the mean squared error between inputs and reconstructions."""
    # STUDENT TASK 3: return the mean of (inputs - reconstructions) ** 2
    raise NotImplementedError("Return the scalar reconstruction MSE.")


In [ ]:
mse_inputs = torch.zeros(2, 2)
mse_reconstructions = torch.full((2, 2), 0.5)
reconstruction_probe_value = round(
    reconstruction_mse(mse_inputs, mse_reconstructions).item(), 6
)
reconstruction_probe_contract = abs(reconstruction_probe_value - 0.25) < 1e-4
assert reconstruction_probe_contract, "Each squared error is 0.25; their mean is 0.25."
print("Task 3 checks passed. Reconstruction MSE probe:", reconstruction_probe_value)


## Provided: training and evaluation rails

The contrastive objective calls your `info_nce_loss`, the autoencoder calls your
`reconstruction_mse`, and every evaluation calls your `knn_predict`. Encoders are
built from a width you control in task 4.


In [ ]:
def build_encoder(width, embedding_dim=128):
    return nn.Sequential(
        nn.Conv2d(3, width, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(width, width, 3, padding=1), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        nn.Linear(width, embedding_dim),
    )


def augment(batch, kind):
    if kind == "geometry":
        if random.random() < 0.5:
            batch = torch.flip(batch, dims=[3])
        shift = random.randint(-2, 2)
        return torch.roll(batch, shifts=shift, dims=2)
    if kind == "color":
        scale = 0.5 + torch.rand(batch.shape[0], 1, 1, 1, device=batch.device)
        gray = batch.mean(dim=1, keepdim=True)
        return scale * (0.5 * batch + 0.5 * gray)
    return batch


@dataclass(frozen=True)
class RepresentationExperiment:
    objective: str = "contrastive"
    encoder_width: int = 64
    augmentation: str = "geometry"
    projection_dim: int = 128
    learning_rate: float = 1e-3
    batch_size: int = DEFAULT_BATCH_SIZE
    epochs: int = EPOCHS
    seed: int = SEED


def changed_config_fields(reference, candidate):
    reference_fields = asdict(reference)
    candidate_fields = asdict(candidate)
    return {
        name for name in reference_fields
        if reference_fields[name] != candidate_fields[name]
    }


def history_is_complete(record):
    history = record["history"]
    epochs = record["config"]["epochs"]
    return all(
        len(history[metric]) == epochs
        for metric in (
            "train_loss", "train_accuracy",
            "validation_loss", "validation_accuracy",
        )
    )


@torch.no_grad()
def extract_features(encoder, loader):
    encoder.eval()
    features, labels = [], []
    for images, targets in loader:
        features.append(encoder(images.to(DEVICE)).cpu())
        labels.append(targets)
    return torch.cat(features), torch.cat(labels)


def knn_accuracy(encoder, reference_loader, query_loader):
    # Freeze the encoder, embed both splits, and let your knn_predict label the
    # queries by their nearest reference neighbours. This is the readout that
    # says whether a representation organized the classes.
    reference_features, reference_labels = extract_features(encoder, reference_loader)
    query_features, query_labels = extract_features(encoder, query_loader)
    predictions = knn_predict(
        reference_features, reference_labels, query_features, k=5
    )
    return (predictions == query_labels).float().mean().item()


def run_experiment(label, config):
    # Train one encoder: the contrastive objective calls your info_nce_loss, the
    # autoencoder objective calls your reconstruction_mse, and each epoch is scored
    # with the kNN probe above. Returns the config and full history.
    reset_seed(config.seed)
    encoder = build_encoder(config.encoder_width, config.projection_dim).to(DEVICE)
    fit_loader = seeded_loader(
        train_set, batch_size=config.batch_size, shuffle=True, seed=config.seed
    )
    reference_loader = seeded_loader(
        train_set, batch_size=config.batch_size, shuffle=False, seed=config.seed
    )
    validation_loader = seeded_loader(
        validation_set, batch_size=config.batch_size, shuffle=False,
        seed=config.seed,
    )
    if config.objective == "contrastive":
        projection = nn.Linear(config.projection_dim, config.projection_dim).to(DEVICE)
        parameters = list(encoder.parameters()) + list(projection.parameters())
    else:
        decoder = nn.Linear(config.projection_dim, 3 * 32 * 32).to(DEVICE)
        parameters = list(encoder.parameters()) + list(decoder.parameters())
    optimizer = torch.optim.AdamW(parameters, lr=config.learning_rate)
    history = {
        "train_loss": [], "train_accuracy": [],
        "validation_loss": [], "validation_accuracy": [],
    }
    for epoch in range(1, config.epochs + 1):
        encoder.train()
        epoch_loss = batches = 0.0
        for images, _ in fit_loader:
            images = images.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            if config.objective == "contrastive":
                z1 = projection(encoder(augment(images, config.augmentation)))
                z2 = projection(encoder(augment(images, config.augmentation)))
                loss = info_nce_loss(z1, z2, temperature=0.5)
            else:
                reconstruction = decoder(encoder(images)).view(images.shape)
                loss = reconstruction_mse(reconstruction, images)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); batches += 1
        train_loss = epoch_loss / batches
        validation_accuracy = knn_accuracy(
            encoder, reference_loader, validation_loader
        )
        train_accuracy = knn_accuracy(encoder, reference_loader, reference_loader)
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["validation_loss"].append(train_loss)
        history["validation_accuracy"].append(validation_accuracy)
        print(f"{label:>26} | epoch {epoch}/{config.epochs} | "
              f"loss {train_loss:.3f} | val kNN {validation_accuracy:.2%}")
    record = {"label": label, "config": asdict(config), "history": history}
    del encoder
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return record


baseline_config = RepresentationExperiment()
baseline_result = run_experiment("contrastive baseline", baseline_config)
anchor_config = replace(baseline_config, objective="autoencoder")
anchor_result = run_experiment("autoencoder anchor", anchor_config)


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 4</div>
<div style="font-weight:700;font-size:1.2rem;">Design an encoder</div>
</div>


Choose `encoder_width` so the encoder has **100,000 to 900,000** trainable
parameters. The encoder has `9 * width^2 + 157 * width + 128` parameters, so
wider encoders add capacity and cost.


In [ ]:
# STUDENT TASK 4: pick a positive encoder_width inside the budget.
encoder_width = 0

if encoder_width <= 0:
    raise ValueError("Choose a positive encoder_width.")
learner_config = replace(baseline_config, encoder_width=encoder_width)
learner_encoder = build_encoder(learner_config.encoder_width, learner_config.projection_dim)
learner_encoder_parameter_count = trainable_parameter_count(learner_encoder)
learner_parameter_count = learner_encoder_parameter_count
if not (100_000 <= learner_parameter_count <= 900_000):
    raise ValueError("Choose encoder_width so the encoder has 100,000-900,000 parameters.")
print(f"learner encoder parameters: {learner_parameter_count:,}")


In [ ]:
learner_result = run_experiment("learner encoder design", learner_config)
for record in (baseline_result, anchor_result, learner_result):
    epochs = np.arange(1, len(record["history"]["validation_accuracy"]) + 1)
    plt.plot(epochs, record["history"]["validation_accuracy"],
             marker="o", label=record["label"])
plt.xlabel("epoch"); plt.ylabel("val kNN accuracy"); plt.ylim(0, 1)
plt.legend(); plt.tight_layout(); plt.show()


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 5</div>
<div style="font-weight:700;font-size:1.2rem;">Choose one augmentation</div>
</div>


Positive pairs define the invariances a contrastive encoder learns. Predict what
each choice encourages the encoder to ignore, then choose exactly one augmentation
relative to the geometry baseline: `color`, `geometry`, or `none`. The contract
permits exactly one changed field.


In [ ]:
# STUDENT TASK 5: choose "color", "geometry", or "none".
AUGMENTATION_CHOICE = ""

AUGMENTATION_OVERRIDES = {
    "color": {"augmentation": "color"},
    "geometry": {"augmentation": "geometry"},
    "none": {"augmentation": "none"},
}
AUGMENTATION_FIELDS = {name: "augmentation" for name in AUGMENTATION_OVERRIDES}
AUGMENTATION_LABELS = {
    "color": "chosen augmentation: color",
    "geometry": "chosen augmentation: geometry",
    "none": "chosen augmentation: none",
}
if AUGMENTATION_CHOICE not in AUGMENTATION_OVERRIDES:
    raise ValueError("Choose color, geometry, or none.")
if AUGMENTATION_OVERRIDES[AUGMENTATION_CHOICE]["augmentation"] == baseline_config.augmentation:
    raise ValueError("Choose an augmentation different from the baseline geometry setting.")
variation_config = replace(
    baseline_config, **AUGMENTATION_OVERRIDES[AUGMENTATION_CHOICE]
)
assert changed_config_fields(baseline_config, variation_config) == {"augmentation"}


In [ ]:
variation_result = run_experiment(
    AUGMENTATION_LABELS[AUGMENTATION_CHOICE], variation_config
)
for record in (baseline_result, variation_result):
    epochs = np.arange(1, len(record["history"]["validation_accuracy"]) + 1)
    plt.plot(epochs, record["history"]["validation_accuracy"],
             marker="o", label=record["label"])
plt.xlabel("epoch"); plt.ylabel("val kNN accuracy"); plt.ylim(0, 1)
plt.legend(); plt.tight_layout(); plt.show()


## Pause and reflect (ungraded)

Which objective produced better-organized neighborhoods, and which augmentation
most changed the encoder's invariances? Name one caveat that keeps the conclusion
tentative and one single-factor experiment you would run next. This is a place to
pause, not a response field.


## Report values

Run the cell below after completing all five tasks and all four runs. R1-R3
verify your three functions on fixed probes. R4 is **your** valid encoder
design's parameter count in the 100,000-900,000 range. R5 verifies the disjoint
split, single-factor comparisons, and complete histories. Enter R1-R5 in the
first Lab 4 submission problem. No measured probe accuracy is submitted.


In [ ]:
run_config_pairs = (
    (baseline_result, baseline_config),
    (anchor_result, anchor_config),
    (learner_result, learner_config),
    (variation_result, variation_config),
)

workflow_contract = int(
    set(train_indices).isdisjoint(validation_indices)
    and len({record["label"] for record, _ in run_config_pairs}) == 4
    and all(
        record["config"] == asdict(config)
        for record, config in run_config_pairs
    )
    and all(history_is_complete(record) for record, _ in run_config_pairs)
    and changed_config_fields(baseline_config, anchor_config) == {"objective"}
    and changed_config_fields(baseline_config, learner_config) == {"encoder_width"}
    and changed_config_fields(baseline_config, variation_config) == {"augmentation"}
    and learner_parameter_count == learner_encoder_parameter_count
    and 100_000 <= learner_parameter_count <= 900_000
)

probe_infonce_loss = (
    round(infonce_probe_value, 6) if infonce_probe_contract else -1.0
)
probe_knn_correct = knn_probe_correct if knn_probe_contract else -1
probe_reconstruction_mse = (
    round(reconstruction_probe_value, 6) if reconstruction_probe_contract else -1.0
)

report_values = {
    "R1: InfoNCE loss on the fixed one-hot probe": probe_infonce_loss,
    "R2: correct kNN predictions on the fixed probe": probe_knn_correct,
    "R3: reconstruction MSE on the fixed probe": probe_reconstruction_mse,
    "R4: learner-designed encoder parameter count": learner_parameter_count,
    "R5: experimental workflow contract": workflow_contract,
}
assert abs(probe_infonce_loss - 0.340753) < 1e-4
assert probe_knn_correct == 3
assert abs(probe_reconstruction_mse - 0.25) < 1e-4
assert 100_000 <= learner_parameter_count <= 900_000
assert workflow_contract == 1

print("LAB 4 REPORT VALUES")
for label, value in report_values.items():
    print(f"{label}: {value}")
